# Transport Dimension Data - Gold Layer

## Objective
Extract and deduplicate unique transport mode data from silver.silver_multimodal to build a standardized Transport Dimension Gold Delta table (gold.gold_multimodal_dim_transport).

## Data Flow
silver.silver_multimodal → Spark SQL / DataFrame → gold.gold_multimodal_dim_transport

## Source
The underlying data comes from the multimodal transport network API.

## Input
Silver Delta table: silver.silver_multimodal

## Output
Gold Delta table: gold.gold_multimodal_dim_transport

## Gold Layer Principle
The Gold layer delivers curated, dimensional models and business-level aggregations ready for reporting and analytics. This pipeline isolates unique transport modes and generates surrogate keys to maintain a clean dimension table with strict entity integrity.

## Processing Steps
1. **Load Silver Data:** Read silver.silver_multimodal into PySpark.
2. **Extract Dimension:** Query distinct transport modes and generate surrogate transport keys.
3. **Write to Gold:** Persist deduplicated dataset to gold.gold_multimodal_dim_transport Delta table.

In [0]:
# Load data from silver schema
df_silver_multimodal=spark.table('workspace.silver.silver_multimodal')

In [0]:
# display the dataframe 
df_silver_multimodal.display()

# BUSINESS TRANSFORMATION AND MODELING

In [0]:
# Extract  Dimension transport Data
query_dim_transport = """
SELECT DISTINCT
    CAST(ABS(HASH(transport_mode)) AS INT) AS transport_key,
    transport_mode
FROM workspace.silver.silver_multimodal
"""

df_dim_transport = spark.sql(query_dim_transport)

In [0]:
# Display df_dim_transport
df_dim_transport.display()

# WRITING GOLD TABLE

In [0]:
df_dim_transport\
    .write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true") \
        .saveAsTable("gold.gold_multimodal_dim_transport")

# CHECKING THE GOLD TABLE

In [0]:
%sql 
SELECT * 
FROM workspace.gold.gold_multimodal_dim_transport
LIMIT 10 